# Agentic AI, Day 3 Lab: Reasoning, Planning, and ReAct

For two days you wired the control flow yourself. Today the model takes the wheel. You will build up to **ReAct**, the loop where the model reasons, calls a tool, observes the result, and decides the next step entirely on its own.

We get there in three steps:

1. **Reasoning:** make the model think before it answers (chain of thought, self-consistency).
2. **Planning:** have the model break a goal into its own steps.
3. **ReAct:** reason and act in a loop, until the model decides it is done.

Everything runs locally on Ollama, no cloud keys. We reload the Day 1 tools first so this notebook stands on its own, then build each pattern.

Run the cells in order, top to bottom.

## Setup and a quick reload

Same setup as before: Ollama running, a tool-capable model (`llama3.1:8b` or `qwen2.5:7b`), and the `ollama` client. The cell below reloads `ask`, `get_weather`, and `calculate` from Day 1.

**If your machine does not have `llama3.1`, change `MODEL` to a model you have pulled.**

In [3]:
import ollama, ast, operator
from collections import Counter

MODEL = "llama3.1:8b"   # change to a tool-capable model you have pulled

def ask(prompt: str) -> str:
    """Send one prompt to the model and return the reply text."""
    r = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}])
    return r.message.content

# ---- tools from Day 1 ----
def get_weather(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: The name of the city, for example "Pune".
    """
    data = {"pune": "31C, clear sky", "mumbai": "33C, humid",
            "delhi": "29C, hazy", "bangalore": "26C, light rain"}
    return data.get(city.lower(), f"No weather data for {city}")

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}
def _ev(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_ev(node.left), _ev(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_ev(node.operand))
    raise ValueError("Unsupported expression")
def calculate(expression: str) -> str:
    """Evaluate a basic arithmetic expression. Supports + - * / ** and parentheses.

    Args:
        expression: A math expression, for example "33 - 31".
    """
    return str(_ev(ast.parse(expression, mode="eval").body))

print("Day 1 tools loaded. Using MODEL =", MODEL)

Day 1 tools loaded. Using MODEL = llama3.1:8b


## Milestone 1: Chain of Thought

The cheapest accuracy trick there is: ask the model to reason through the steps before answering, instead of blurting the first thing.

We will ask the same word problem two ways and compare. A model generates one token at a time using everything before it, so when the reasoning is on the page, it guides the final answer.

In [4]:
problem = ("A shop sells pens at 12 rupees each. I buy 7 pens and pay "
           "with a 100 rupee note. How much change do I get?")

snap = ask(problem + " Reply with only the final number.")
cot  = ask(problem + " Think step by step, then give the final answer.")

print("SNAP answer:")
print(snap)
print("\nSTEP-BY-STEP answer:")
print(cot)

SNAP answer:
88

STEP-BY-STEP answer:
To find out how much change you'll get, let's break down the process:

1. You buy 7 pens at 12 rupees each.
2. First, calculate the total cost of the pens: 7 pens × 12 rupees/pens = 84 rupees
3. You pay with a 100 rupee note.
4. To find out how much change you'll get, subtract the total cost from the amount paid: 100 rupees (payment) - 84 rupees (total cost) = 16 rupees

Therefore, the change you'll receive is **16 rupees**.


**What you should see:** the snap answer is just a number (and on a small model it is sometimes wrong). The step-by-step version shows the working: 7 times 12 is 84, then 100 minus 84 is 16, and the final answer is usually correct.

**Your turn:** make the problem harder (add a discount, or a second item) and run it again. The gap between snap and step-by-step usually widens as the problem gets harder.

## Milestone 2: Self-consistency

Chain of thought plus a vote. A model can reason its way to a wrong answer once, but it is less likely to make the same mistake five times. So we sample several reasonings with a little randomness, pull the final answer out of each, and take the majority.

We ask the model to end with `Answer: <number>` so the result is easy to parse.

In [5]:
def final_answer(text: str) -> str:
    """Pull the final answer out of a reasoning response."""
    for line in reversed(text.splitlines()):
        if "answer:" in line.lower():
            return line.split(":", 1)[1].strip()
    return text.strip().splitlines()[-1].strip()   # fallback: last line

question = ("A train travels 150 km in 2.5 hours. What is its average "
            "speed in km per hour? Think step by step and end with "
            "'Answer: <number>'.")

def sample() -> str:
    r = ollama.chat(model=MODEL,
                    messages=[{"role": "user", "content": question}],
                    options={"temperature": 0.8})   # add variety between samples
    return final_answer(r.message.content)

votes = [sample() for _ in range(5)]
winner = Counter(votes).most_common(1)[0][0]
print("samples:", votes)
print("majority answer:", winner)

samples: ['60', '60', '60', '60', '60']
majority answer: 60


**What you should see:** five answers, most of which should be 60, and a majority answer of 60. If one sample slips up, the vote still lands on the right value.

**The trade-off:** self-consistency costs several calls instead of one. Use it when an answer really matters, not for everyday questions.

## Milestone 3: Make a plan

Planning is decomposition: the model breaks a big goal into an ordered list of smaller steps before doing any of them. This is the same idea as prompt chaining from Day 1, except now the model writes the chain.

In [6]:
def make_plan(goal: str) -> str:
    """Ask the model to break a goal into a short numbered list of steps."""
    return ask(
        "Break this goal into a short numbered list of concrete steps. "
        "Keep it to 4 steps or fewer.\n\n"
        f"Goal: {goal}"
    )

goal = "Compare the current weather in Pune and Mumbai and say which is hotter."
print(make_plan(goal))

Here's a short numbered list of concrete steps:

1. **Find current temperature data for Pune**: Look up the current temperature in Pune using online sources such as AccuWeather, Weather.com, or the Indian Meteorological Department website.
2. **Find current temperature data for Mumbai**: Similarly, look up the current temperature in Mumbai using online sources.
3. **Compare temperatures**: Compare the current temperatures of Pune and Mumbai to determine which city is currently hotter.
4. **Determine "hotter" based on a clear criterion**: Decide how you will define "hotter", e.g. whether it's just higher temperature, or also considering factors like humidity.


**What you should see:** a short numbered plan, something like: get Pune weather, get Mumbai weather, compare the temperatures, state which is hotter.

A plain plan like this is **static**: it is decided up front and cannot adapt if a step reveals something surprising. In the next milestone we move to ReAct, where the model plans and acts together, so it can adapt as it goes.

## Milestone 4: The ReAct loop

ReAct stands for Reason and Act. The model thinks about the next step, calls one tool, sees the result, then thinks again, looping until it can answer.

If you look closely, this is the Day 1 tool loop with three additions:
- a **system prompt** that tells the model to reason before acting,
- a clear **done condition** (no tool call means the model is finished),
- and a **step budget** so it cannot loop forever.

The guards from Day 1 (unknown tool, tool errors) are kept so the loop stays robust.

In [13]:
SYSTEM = (
    "You are a careful agent. Think briefly about what to do next, "
    "use a tool when you need a fact or a calculation."
)

def react(question, tools_list, tools_map, max_steps=8, verbose=True):
    """Reason, act, observe, repeat, until the model is done or the budget runs out."""
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": question}]

    for step in range(max_steps):
        res = ollama.chat(model=MODEL, messages=msgs, tools=tools_list)
        msgs.append(res.message)

        if not res.message.tool_calls:
            return res.message.content            # the model decided it is done

        for c in res.message.tool_calls:
            name = c.function.name
            args = c.function.arguments or {}
            if verbose:
                print(f"[step {step}] act: {name}({args})")

            fn = tools_map.get(name)
            if fn is None:
                out = f"Error: no tool named '{name}'."
            else:
                try:
                    out = str(fn(**args))
                except Exception as e:
                    out = f"Error running {name}: {e}"

            if verbose:
                print(f"          obs: {out}")
            msgs.append({"role": "tool", "content": out})

    return "Stopped: reached the step budget without finishing."

# a simple single-tool run to confirm the loop works
print(react("Compare the current weather in Pune and Mumbai and say which is hotter.", [get_weather], {"get_weather": get_weather}))

[step 0] act: get_weather({'city': 'Pune'})
          obs: 31C, clear sky
[step 0] act: get_weather({'city': 'Mumbai'})
          obs: 33C, humid
Based on the tool call responses, it appears that Mumbai is currently hotter than Pune. The current temperature in Mumbai is 33°C (humid) while in Pune it is 31°C (clear sky). Therefore, Mumbai is hotter at this moment.


**What you should see:** one act line calling `get_weather`, one obs line with the result, then a final sentence answering the question. The model called the tool, observed, and then decided it was done.

**Your turn:** ask a question that needs no tool at all, like "what is a good name for a pet turtle". The model should answer directly with zero tool calls, which is the done condition firing immediately.

## Milestone 5: A multi-step task

This is where ReAct earns its name. Ask a question that needs several tool calls in sequence, where the model has to reason between them. Our weather tool returns a temperature like `31C`, so to compare two cities the model must fetch both, then calculate the difference.

In [ ]:
answer = react(
    "Is it hotter in Pune or Mumbai right now, and by how many degrees?",
    [get_weather, calculate],
    {"get_weather": get_weather, "calculate": calculate},
)
print("\nFINAL:", answer)

NameError: name 'react' is not defined

**What you should see:** the trace shows `get_weather` called for Pune, then for Mumbai, then `calculate` for the difference, then a final answer naming the hotter city and the gap. The model planned and executed that sequence on its own. That is a real agent at work.

Local models vary. If it does not chain all three calls, run it again or switch to `qwen2.5:7b`. Handling that unreliability is the point of the next milestone.

## Milestone 6: Guard it

An agent that cannot tell when it is finished either stops too early or loops forever. The step budget is your hard safety net. Here we give the agent a goal it cannot really complete with the tools it has, and a tight budget, to watch the guard work.

In [ ]:
# a goal with no suitable tool, plus a tight budget to show the safety net
result = react(
    "Book me a complete week-long trip to Japan with exact flight numbers.",
    [get_weather, calculate],
    {"get_weather": get_weather, "calculate": calculate},
    max_steps=3,
)
print("\nRESULT:", result)

**What you should see:** the agent either answers that it cannot do this with the tools it has, or it spins for a few steps and then hits the budget and returns the "Stopped" message. Either way it does not loop forever and it does not crash. That is the goal: stop cleanly and report, rather than run away.

**Two lessons from this milestone:**
- A **clear goal** helps the model know when it is done. Vague goals cause both early stops and endless loops.
- A **step budget** is non-negotiable. It is the difference between a safe agent and a runaway one.

## You did it

Today the model took the wheel:

- **Chain of thought:** reasoning before answering, for free accuracy.
- **Self-consistency:** sample several reasonings and vote.
- **Planning:** the model writes its own steps.
- **ReAct:** reason, act, observe, repeat, with a done condition and a step budget.
- And you saw it **solve a multi-step task on its own**, and saw the guards keep it safe.

The ReAct loop is the frame that holds everything from the week so far: tools are the actions, routing can pick the approach, reflection can check the result.

### Optional challenges

1. **Self-consistency on ReAct:** run `react` on the same multi-step question three times and majority-vote the final answers.
2. **A progress note:** print a short "still working" line each loop in `react`, so a demo is easy to follow.
3. **Add a tool:** give the agent a `word_count` or `read_file` tool and ask a question that needs it mid-loop.
4. **Compare budgets:** run the multi-step task with `max_steps=2` and with `max_steps=8`, and see how the budget changes the outcome.

### What comes next

Tomorrow, Day 4, we give the agent two things it is missing: **memory**, so it remembers across turns, and **retrieval (RAG)**, so it can look up real knowledge instead of guessing. Today the model can reason and act. Tomorrow it can remember and look things up.